# FastKAN Human-vs-NonHuman: Deep Analysis

This notebook provides a detailed comparison between the standard **MLP (Baseline)** and the **FastKAN** head. We will explore:
1. **Model Architectures & Parameters**
2. **Training Dynamics (Accuracy/Loss)**
3. **Interpretability: Visualizing Learned KAN Curves**
4. **Final Performance Metrics**

In [ ]:
import sys
sys.path.append('src')

import torch
from model import BaselineClassifier, KANClassifier
from utils import get_model_summary
import matplotlib.pyplot as plt
import numpy as np

## 1. Parameter Comparison

At this input size (1280 MobileNetV2 features), KAN has **more** trainable parameters than MLP — not fewer.

Each KAN edge stores:
- A `base_weight` (like a standard linear weight)
- `spline_weights` — RBF basis function coefficients (typically 8 per edge)

So for `FastKAN([1280, 64, 2])`:
- Layer 1 (1280→64): 64 × 1280 × (1 + 8) ≈ 737K params
- Layer 2 (64→2): 2 × 64 × (1 + 8) ≈ 1.1K params

Versus the MLP head `Linear(1280, 2)`: just 2,562 params.

**KAN's advantage here is not parameter count — it's interpretability.** Each edge learns a visualizable univariate function, unlike MLP weights which are uninterpretable scalars.

In [ ]:
baseline = BaselineClassifier()
kan = KANClassifier()

b_total, b_train = get_model_summary(baseline)
k_total, k_train = get_model_summary(kan)

print(f"{'Model':<12} {'Total Params':>15} {'Trainable Params':>18}")
print("-" * 47)
print(f"{'Baseline':<12} {b_total:>15,} {b_train:>18,}")
print(f"{'KAN':<12} {k_total:>15,} {k_train:>18,}")
print()
print(f"KAN head has {k_train / b_train:.0f}x more trainable params than MLP head.")
print("This is expected — KAN spline weights add significant capacity per edge.")
print("The comparison goal is accuracy parity + interpretability, not parameter reduction.")

## 2. Visualizing KAN Interpretability
Unlike MLPs where weights are fixed scalars, KANs learn **univariate functions (curves)** on the edges. Let's visualize what the model learned.

## 3. Save KAN Curves to Disk

Saves a grid of learned activation curves for each output class as PNG files in `results/`.
Each subplot shows how one MobileNetV2 feature dimension gets transformed by the KAN edge.